In [23]:
import gymnasium as gym
import numpy as np
import time
import sys
import os

sys.path.append(os.path.expanduser("~/ambf/surgical_robotics_challenge/scripts"))
import gymnasium as gym
from stable_baselines3.common.evaluation import evaluate_policy
from Insert_env import SRC_insert
import numpy as np
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_checker import check_env
from RL_algo.td3_BC import TD3_BC
from RL_algo.DemoHerReplayBuffer import DemoHerReplayBuffer
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.noise import NormalActionNoise
import pickle
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
import torch
from stable_baselines3 import TD3
from RL_algo.td3_BC import TD3_BC  # 載入您的算法


from surgical_robotics_challenge.utils import interpolation
_original_get_interpolated_x = interpolation.Interpolation.get_interpolated_x
def safe_get_interpolated_x(self, t):
    if t < self._t0: t = self._t0
    elif t > self._tf: t = self._tf
    return _original_get_interpolated_x(self, t)
interpolation.Interpolation.get_interpolated_x = safe_get_interpolated_x
print("🛡️ Monkey Patch 已套用：測試過程將保持穩定。")
# ==========================================


🛡️ Monkey Patch 已套用：測試過程將保持穩定。


In [24]:
# 1. 載入環境參數
env_info_path = "/home/suzixun/SurgicAI/RL/Env_info/Insert_noise_env_info"
if os.path.exists(env_info_path):
    with open(env_info_path, 'rb') as file:
        env_info = pickle.load(file)
    step_size = np.array(env_info["step_size"], dtype=np.float32)
    threshold = np.array(env_info["threshold"], dtype=np.float32)
    episode_steps = int(env_info["max_timestep"])
else:
    # 備用預設值
    step_size = None
    threshold = None
    episode_steps = 200

# 2. 建立測試環境 (開啟 render_mode="human")
# 請確保 SRC_insert 類別已經在前面的 Cell 定義過了
if "TD3_HER_BC_Insert_Test" in gym.envs.registry:
    del gym.envs.registry["TD3_HER_BC_Insert_Test"]

gym.envs.register(id="TD3_HER_BC_Insert_Test", entry_point=SRC_insert, max_episode_steps=episode_steps)

test_env = gym.make(
    "TD3_HER_BC_Insert_Test", 
    render_mode="human",      # <--- 關鍵！開啟 3D 畫面
    reward_type="sparse",     # 保持與訓練一致
    max_episode_step=episode_steps,
    seed=10, 
    step_size=step_size,
    threshold=threshold
)


max episode length is 434
step size is [0.0002     0.0002     0.0002     0.03490658 0.03490658 0.03490658
 0.02      ]
Set random seed
reward type is sparse
Translation threshold: 0.10000000149011612, angle threshold: 0.0872664600610733
Found Common Object Namespace as:  /ambf/env/
psm1_target_ik NAMED OBJECT NOT FOUND
psm1_palm_joint_ik NAMED OBJECT NOT FOUND
psm1_target_fk NAMED OBJECT NOT FOUND
ERROR! Requested Joint Idx of "6" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "7" outside valid range [0 - -1]
psm2_target_ik NAMED OBJECT NOT FOUND
psm2_palm_joint_ik NAMED OBJECT NOT FOUND
psm2_target_fk NAMED OBJECT NOT FOUND
ERROR! Requested Joint Idx of "6" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "7" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "0" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "1" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "2" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "3" outsid

In [25]:
# 3. 載入您訓練好的模型
model_path = "/home/suzixun/SurgicAI/RL/Insert/TD3_HER_BC/sparse/seed_0/final_model"

print(f"📂 正在載入模型: {model_path} ...")

# 嘗試載入 (處理可能的 custom_objects 問題)
try:
    model = TD3_BC.load(model_path, env=test_env)
except Exception as e:
    print(f"⚠️ 使用 TD3_BC 載入失敗 ({e})，嘗試使用標準 TD3 載入...")
    model = TD3.load(model_path, env=test_env)

print("✅ 模型載入成功！開始 5 回合演示...")


📂 正在載入模型: /home/suzixun/SurgicAI/RL/Insert/TD3_HER_BC/sparse/seed_0/final_model ...
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
✅ 模型載入成功！開始 5 回合演示...


In [26]:
# 4. 開始演示迴圈
num_episodes = 20

for ep in range(num_episodes):
    obs, info = test_env.reset()
    done = False
    truncated = False
    total_reward = 0
    step = 0
    
    print(f"\n🎬 Episode {ep + 1} 開始")
    
    while not (done or truncated):
        # deterministic=True 代表「不加雜訊」，展示模型的真實實力
        action, _ = model.predict(obs, deterministic=True)
        
        obs, reward, done, truncated, info = test_env.step(action)
        
        total_reward += reward
        step += 1
        
        # 稍微暫停一下，讓肉眼跟得上動作 (可自行調整)
        time.sleep(0.3) 

    print(f"🏁 Episode {ep + 1} 結束 | Steps: {step} | Reward: {total_reward:.2f}")
    if info.get('is_success'):
        print("✨✨✨ 任務成功 (SUCCESS)！ ✨✨✨")
    else:
        print("💀 任務失敗")
        
    time.sleep(1.0) # 回合間休息

test_env.close()
print("演示結束。")

ERROR! Requested Joint Idx of "0" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "1" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "2" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "3" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "4" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "5" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "6" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "7" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "0" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "1" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "2" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "3" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "4" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "5" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "6" outside valid range [0 - -1]
ERROR! Requested Joint Idx of "7" outside valid range [

TypeError: unsupported operand type(s) for /: 'NoneType' and 'float'

step_size = [0.0005     0.0005     0.0005     0.03490658 0.03490658 0.03490658
 0.05      ]
threshold = [0.5       0.5235988]
episode_steps = 200


Exception in thread Thread-133:
Traceback (most recent call last):
  File "/tmp/ipykernel_261173/2401389482.py", line 33, in safe_get_interpolated_x
  File "/tmp/ipykernel_261173/2401389482.py", line 33, in safe_get_interpolated_x
  File "/tmp/ipykernel_261173/2401389482.py", line 33, in safe_get_interpolated_x
  [Previous line repeated 996 more times]
  File "/tmp/ipykernel_261173/2401389482.py", line 31, in safe_get_interpolated_x
RecursionError: maximum recursion depth exceeded in comparison


reset!!!

🎬 Episode 1 開始
Maximum step reaches
🏁 Episode 1 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 2 開始
Maximum step reaches
🏁 Episode 2 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 3 開始
Maximum step reaches
🏁 Episode 3 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 4 開始
Maximum step reaches
🏁 Episode 4 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 5 開始
Maximum step reaches
🏁 Episode 5 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 6 開始
Maximum step reaches
🏁 Episode 6 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 7 開始
Maximum step reaches
🏁 Episode 7 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 8 開始
Maximum step reaches
🏁 Episode 8 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 9 開始
Maximum step reaches
🏁 Episode 9 結束 | Steps: 200 | Reward: -200.00
💀 任務失敗
reset!!!

🎬 Episode 10 開始


Exception in thread Thread-80:
Traceback (most recent call last):
  File "/usr/lib/python3.8/threading.py", line 932, in _bootstrap_inner
    self.run()
  File "/home/suzixun/.local/lib/python3.8/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.8/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "<string>", line 321, in _run_obj_publishers
RuntimeError: dictionary changed size during iteration


shutdown request: [/src_client] Reason: new node registered with same name
Closing publisher for:  /ambf/env/Plane
Closing publisher for:  /ambf/env/lights/light2
Closing publisher for:  /ambf/env/cameras/cameraL
Closing publisher for:  /ambf/env/cameras/cameraR
Closing publisher for:  /ambf/env/cameras/cameraL2
Closing publisher for:  World
Closing publisher for:  /ambf/env/psm1/baselink
Closing publisher for:  /ambf/env/psm1/toolyawlink
Closing publisher for:  /ambf/env/psm1/Sensor0
Closing publisher for:  /ambf/env/psm1/Actuator0
Closing publisher for:  /ambf/env/psm2/baselink
Closing publisher for:  /ambf/env/psm2/toolyawlink
Closing publisher for:  /ambf/env/psm2/Sensor0
Closing publisher for:  /ambf/env/psm2/Actuator0
Closing publisher for:  /ambf/env/BlackGround
Closing publisher for:  /ambf/env/CameraFrame
Closing publisher for:  /ambf/env/Ledge
Closing publisher for:  /ambf/env/Needle
Closing publisher for:  /ambf/env/Phantom
Closing publisher for:  /ambf/env/Barrier
Closing p

ROSException: publish() to a closed topic